**Phase 1: Environment Setup & Dependencies**  
Installs the required core libraries for vector embeddings, local LLM inference, and Neo4j graph database management.

In [ ]:

!pip install --quiet llama-index llama-index-graph-stores-neo4j llama-index-embeddings-huggingface llama-index-llms-huggingface transformers accelerate bitsandbytes neo4j pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 15.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

**Phase 2: Infrastructure & Model Loading**  
Establishes the secure connection to the Neo4j Aura database and loads the local 7B parameter extraction model (Qwen 2.5) using 4-bit quantization alongside the BGE embedding model.

In [ ]:
import os
import torch
from google.colab import userdata
from transformers import BitsAndBytesConfig
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. Fetch secrets securely
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
neo4j_uri = userdata.get('NEO4J_URI')
neo4j_user = userdata.get('NEO4J_USERNAME')
neo4j_password = userdata.get('NEO4J_PASSWORD')

# 2. Establish Neo4j Connection (Fixed routing issue)
try:
    graph_store = Neo4jPropertyGraphStore(
        username=neo4j_user,
        password=neo4j_password,
        url=neo4j_uri,
        database=None # 👈 This tells Neo4j to just use the default Aura DB
    )
    print("✅ Neo4j Database Connected Successfully!")
except Exception as e:
    print(f"❌ Neo4j Connection Failed: {e}")

# 3. Define 4-bit Quantization Config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# 4. Download and Instantiate Qwen-2.5-7B (Bypassing the Llama-3 License Wall)
print("⏳ Downloading Qwen 2.5... (This will take a few minutes)")
try:
    llm = HuggingFaceLLM(
        model_name="Qwen/Qwen2.5-7B-Instruct", # 👈 Fully open, no gating!
        tokenizer_name="Qwen/Qwen2.5-7B-Instruct",
        context_window=8192,
        max_new_tokens=512,
        model_kwargs={"quantization_config": quantization_config},
        generate_kwargs={"temperature": 0.1, "do_sample": True},
        device_map="auto",
    )
    print("✅ LLM Loaded Successfully into GPU Memory!")
except Exception as e:
    print(f"❌ LLM Loading Failed: {e}")

# 5. Initialize the Embedding Model
try:
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    print("✅ Embedding Model Ready!")
except Exception as e:
    print(f"❌ Embedding Model Failed: {e}")

✅ Neo4j Database Connected Successfully!
⏳ Downloading Qwen 2.5... (This will take a few minutes)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ LLM Loaded Successfully into GPU Memory!
✅ Embedding Model Ready!


**Phase 3: Exploration — Baseline Ingestion (Schema-Based Extraction)**  
⚠️ RESEARCH NOTE: This was our first ingestion attempt using SchemaLLMPathExtractor with a strict type template. We discovered that forcing a compact 7B local model into rigid schema constraints caused unstable relationship generation and partial extraction text drops.

In [ ]:
import nest_asyncio
nest_asyncio.apply()
from llama_index.core import Document
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from llama_index.core import PropertyGraphIndex

# 1. Create a synthetic test dataset mirroring messy financial disclosures
# This ensures we get entity variations like 'Apple', 'Apple Inc.', 'AAPL', 'SVB', and 'Silicon Valley Bank'
documents = [
    Document(text="Apple Inc. reported exceptional earnings for Q3 2025. The consumer electronics giant noted that Apple saw massive growth in services. However, AAPL faced minor supply chain issues in Asia."),
    Document(text="Silicon Valley Bank experienced sudden liquidity constraints in early quarters. Regulators closely monitored SVB during this period. The bank reported severe interest rate risks."),
    Document(text="Microsoft Corp announced investments into AI infrastructure. Microsoft noted that MSFT cloud services grew by 24%. Meanwhile, Apple Inc. remains a key competitor in consumer AI space."),
    Document(text="SVB Financial Group filed disclosures regarding their asset-liability matching strategy. Interest rate risks heavily impacted Silicon Valley Bank's capital ratios.")
]
print(f"📋 Prepared {len(documents)} dense text documents with embedded entity variants.")

# 2. Define the extraction schema for the Graph
# We give the LLM clear boundaries on types, but we set strict=False so it acts messy with text names
entities = ["Company", "RiskFactor", "FinancialMetric"]
relations = ["HAS_RISK", "REPORTED_METRIC", "COMPETES_WITH"]

kg_extractor = SchemaLLMPathExtractor(
    llm=llm,
    possible_entities=entities,
    possible_relations=relations,
    strict=False # Allows the LLM to pull varied text values instead of forcing strict string formats
)

# 3. Process the documents and build the "Messy Control Graph" in Neo4j
print("⏳ Running entity extraction via Qwen... Populating your messy Neo4j Graph. (Takes 1-2 mins)")
try:
    index = PropertyGraphIndex.from_documents(
        documents,
        property_graph_store=graph_store,
        kg_extractors=[kg_extractor],
        embed_model=embed_model
    )
    print("✅ Messy Control Graph successfully created and saved in Neo4j!")
except Exception as e:
    print(f"❌ Graph Extraction Failed: {e}")

📋 Prepared 4 dense text documents with embedded entity variants.
⏳ Running entity extraction via Qwen... Populating your messy Neo4j Graph. (Takes 1-2 mins)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Messy Control Graph successfully created and saved in Neo4j!


In [ ]:
import nest_asyncio
nest_asyncio.apply()

import time
from neo4j import GraphDatabase
from google.colab import userdata
from llama_index.core import Document, PropertyGraphIndex
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor

# 1. Initialize the Neo4j driver connection explicitly
neo4j_uri = userdata.get('NEO4J_URI')
neo4j_user = userdata.get('NEO4J_USERNAME')
neo4j_password = userdata.get('NEO4J_PASSWORD')

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

# 2. Clear out the failed chunks so we start fresh
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("🧹 Cleared old empty database chunks.")

# 3. Re-initialize our messy text corpus
documents = [
    Document(text="Apple Inc. reported exceptional earnings for Q3 2025. The consumer electronics giant noted that Apple saw massive growth in services. However, AAPL faced minor supply chain issues in Asia."),
    Document(text="Silicon Valley Bank experienced sudden liquidity constraints in early quarters. Regulators closely monitored SVB during this period. The bank reported severe interest rate risks."),
    Document(text="Microsoft Corp announced investments into AI infrastructure. Microsoft noted that MSFT cloud services grew by 24%. Meanwhile, Apple Inc. remains a key competitor in consumer AI space."),
    Document(text="SVB Financial Group filed disclosures regarding their asset-liability matching strategy. Interest rate risks heavily impacted Silicon Valley Bank's capital ratios.")
]

# 4. Use the robust path extractor (Fixed Argument!)
kg_extractor = SimpleLLMPathExtractor(
    llm=llm,
    max_paths_per_chunk=5 # <--- Changed from triplets to paths
)

print("⏳ Running robust entity extraction via Qwen... (Takes ~1 minute)")
try:
    index = PropertyGraphIndex.from_documents(
        documents,
        property_graph_store=graph_store,
        kg_extractors=[kg_extractor],
        embed_model=embed_model
    )
    print("✅ Knowledge Graph successfully populated with real entities!")
except Exception as e:
    print(f"❌ Extraction Failed: {e}")

🧹 Cleared old empty database chunks.
⏳ Running robust entity extraction via Qwen... (Takes ~1 minute)
✅ Knowledge Graph successfully populated with real entities!


**Phase 4: Exploration — Algorithm 1: Lexical String Resolution**  
⚠️ RESEARCH NOTE: This step runs a deterministic Levenshtein distance string-matching algorithm. We observed that while it successfully handles slight typographical differences, it is fundamentally incapable of resolving symbolic acronym pairs like SVB and Silicon Valley Bank due to a near-zero string similarity score.

In [ ]:
import time
from neo4j import GraphDatabase
from google.colab import userdata

# 1. Initialize the Neo4j driver connection explicitly
neo4j_uri = userdata.get('NEO4J_URI')
neo4j_user = userdata.get('NEO4J_USERNAME')
neo4j_password = userdata.get('NEO4J_PASSWORD')

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

def run_lexical_resolution():
    print("🚀 Starting Algorithm 1: Lexical Matching (String Edit Distance)...")
    start_time = time.time()

    # This Cypher query finds pairs of 'entity' nodes, computes their Levenshtein similarity based on their text (e.id),
    # and uses Neo4j's built-in APOC refactoring library to safely merge them.
    cypher_merge_query = """
    MATCH (e1:entity), (e2:entity)
    WHERE id(e1) > id(e2)
    WITH e1, e2, apoc.text.levenshteinSimilarity(e1.id, e2.id) AS similarityScore
    WHERE similarityScore > 0.80
    CALL apoc.refactor.mergeNodes([e1, e2], {properties: "combine", mergeRels: true}) YIELD node
    RETURN node.id[0] AS merged_name, e1.id AS original_1, e2.id AS original_2, similarityScore
    """

    with driver.session() as session:
        result = session.run(cypher_merge_query)
        records = list(result)

    end_time = time.time()
    duration = end_time - start_time

    print(f"⏱️ Execution completed in {duration:.4f} seconds.")
    if len(records) == 0:
        print("ℹ️ No lexical duplicates were close enough to merge (> 0.80 similarity).")
    else:
        print(f"✨ Successfully merged {len(records)} sets of lexically identical nodes:")
        for record in records:
            print(f"   - Merged '{record['original_1']}' and '{record['original_2']}' (Score: {record['similarityScore']:.2f})")

    return duration

# Run the algorithm
lexical_duration = run_lexical_resolution()

🚀 Starting Algorithm 1: Lexical Matching (String Edit Distance)...


⏱️ Execution completed in 1.8360 seconds.
ℹ️ No lexical duplicates were close enough to merge (> 0.80 similarity).


Phase 3: The Fix (Algorithm 2 - Semantic Clustering)

In [ ]:
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def run_semantic_resolution():
    print("🚀 Starting Algorithm 2: Semantic Embedding Resolution...")
    start_time = time.time()

    # 1. Fetch all entity names and their Neo4j element IDs
    fetch_query = "MATCH (e:entity) RETURN elementId(e) AS el_id, e.id AS name"
    with driver.session() as session:
        records = list(session.run(fetch_query))

    if not records:
        print("❌ No entities found to cluster.")
        return 0

    nodes = [{"el_id": r["el_id"], "name": r["name"]} for r in records]
    names = [n["name"] for n in nodes]
    print(f"📊 Analyzing {len(names)} unique entity names...")

    # 2. Generate Vector Embeddings for all names
    # (embed_model is already loaded in your memory from Phase 1)
    embeddings = [embed_model.get_text_embedding(name) for name in names]
    embeddings_matrix = np.array(embeddings)

    # 3. Calculate Cosine Similarity between all pairs
    similarity_matrix = cosine_similarity(embeddings_matrix)

    # 4. Find pairs with > 0.85 similarity and merge them
    merge_count = 0
    merged_ids = set() # Track what we merge to avoid double-merging in this loop

    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if similarity_matrix[i][j] > 0.85:
                el_id_1, name_1 = nodes[i]["el_id"], nodes[i]["name"]
                el_id_2, name_2 = nodes[j]["el_id"], nodes[j]["name"]

                # Skip if either node was already merged
                if el_id_1 in merged_ids or el_id_2 in merged_ids:
                    continue

                print(f"   🔗 Semantic Match Found! '{name_1}' & '{name_2}' (Score: {similarity_matrix[i][j]:.2f})")

                # Cypher query to physically merge them in Neo4j
                merge_query = """
                MATCH (e1), (e2)
                WHERE elementId(e1) = $id1 AND elementId(e2) = $id2
                CALL apoc.refactor.mergeNodes([e1, e2], {properties: "combine", mergeRels: true}) YIELD node
                RETURN node
                """
                with driver.session() as session:
                    session.run(merge_query, id1=el_id_1, id2=el_id_2)

                merged_ids.add(el_id_1)
                merged_ids.add(el_id_2)
                merge_count += 1

    end_time = time.time()
    duration = end_time - start_time
    print(f"\n⏱️ Execution completed in {duration:.4f} seconds.")
    print(f"✨ Successfully merged {merge_count} semantic duplicate pairs.")

    return duration

# Execute Algorithm 2
semantic_duration = run_semantic_resolution()

🚀 Starting Algorithm 2: Semantic Embedding Resolution...
📊 Analyzing 22 unique entity names...
   🔗 Semantic Match Found! 'Severe interest rate risks' & 'Interest rate risks' (Score: 0.93)

⏱️ Execution completed in 0.8402 seconds.
✨ Successfully merged 1 semantic duplicate pairs.


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("🔍 Diagnostics: Exposing the Vector Math")

# The exact strings that failed to merge
test_pairs = [
    ("Microsoft", "Microsoft corp"),
    ("Silicon valley bank", "Svb financial group")
]

for word1, word2 in test_pairs:
    # 1. Convert words to vectors
    vec1 = embed_model.get_text_embedding(word1)
    vec2 = embed_model.get_text_embedding(word2)

    # 2. Calculate their exact cosine similarity
    score = cosine_similarity([vec1], [vec2])[0][0]

    print(f"Similarity between '{word1}' & '{word2}': {score:.3f}")

🔍 Diagnostics: Exposing the Vector Math
Similarity between 'Microsoft' & 'Microsoft corp': 0.846
Similarity between 'Silicon valley bank' & 'Svb financial group': 0.719


Phase 3: Algorithm 2 (Tuned for 0.71 Threshold)

In [ ]:
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def run_semantic_resolution_tuned():
    print("🚀 Starting Algorithm 2: Semantic Embedding Resolution (Tuned Threshold: 0.71)...")
    start_time = time.time()

    # 1. Fetch all entity names and their Neo4j element IDs
    fetch_query = "MATCH (e:entity) RETURN elementId(e) AS el_id, e.id AS name"
    with driver.session() as session:
        records = list(session.run(fetch_query))

    if not records:
        print("❌ No entities found to cluster.")
        return 0

    nodes = [{"el_id": r["el_id"], "name": r["name"]} for r in records]
    names = [n["name"] for n in nodes]

    # 2. Generate Vector Embeddings for all names
    embeddings = [embed_model.get_text_embedding(name) for name in names]
    embeddings_matrix = np.array(embeddings)

    # 3. Calculate Cosine Similarity between all pairs
    similarity_matrix = cosine_similarity(embeddings_matrix)

    # 4. Find pairs with > 0.71 similarity (Our new empirically tuned threshold!)
    merge_count = 0
    merged_ids = set()

    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if similarity_matrix[i][j] > 0.71: # <--- LOWERED THRESHOLD
                el_id_1, name_1 = nodes[i]["el_id"], nodes[i]["name"]
                el_id_2, name_2 = nodes[j]["el_id"], nodes[j]["name"]

                # Skip if either node was already merged
                if el_id_1 in merged_ids or el_id_2 in merged_ids:
                    continue

                print(f"   🔗 Semantic Match Found! '{name_1}' & '{name_2}' (Score: {similarity_matrix[i][j]:.3f})")

                # Cypher query to physically merge them in Neo4j
                merge_query = """
                MATCH (e1), (e2)
                WHERE elementId(e1) = $id1 AND elementId(e2) = $id2
                CALL apoc.refactor.mergeNodes([e1, e2], {properties: "combine", mergeRels: true}) YIELD node
                RETURN node
                """
                with driver.session() as session:
                    session.run(merge_query, id1=el_id_1, id2=el_id_2)

                merged_ids.add(el_id_1)
                merged_ids.add(el_id_2)
                merge_count += 1

    end_time = time.time()
    duration = end_time - start_time
    print(f"\n⏱️ Execution completed in {duration:.4f} seconds.")
    print(f"✨ Successfully merged {merge_count} semantic duplicate pairs.")

    return duration

# Execute the tuned algorithm
semantic_duration = run_semantic_resolution_tuned()

🚀 Starting Algorithm 2: Semantic Embedding Resolution (Tuned Threshold: 0.71)...


ValidationError: 1 validation error for EmbeddingEndEvent
chunks.0
  Input should be a valid string [type=string_type, input_value=['Severe interest rate ri..., 'Interest rate risks'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [ ]:
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def run_semantic_resolution_safe():
    print("🚀 Starting Robust Algorithm 2: Semantic Resolution (Threshold: 0.71)...")
    start_time = time.time()

    # 1. Fetch all entity names and handles list mutations gracefully
    fetch_query = "MATCH (e:entity) RETURN elementId(e) AS el_id, e.id AS name"
    with driver.session() as session:
        records = list(session.run(fetch_query))

    if not records:
        print("❌ No entities found to cluster.")
        return 0

    nodes = []
    names = []
    for r in records:
        raw_name = r["name"]
        # Crucial Fix: If APOC turned the ID into a list, extract the first string element
        if isinstance(raw_name, list):
            clean_name = str(raw_name[0])
        else:
            clean_name = str(raw_name)

        nodes.append({"el_id": r["el_id"], "name": clean_name})
        names.append(clean_name)

    print(f"📊 Analyzing {len(names)} unique entity names...")

    # 2. Generate Vector Embeddings safely using verified strings
    embeddings = [embed_model.get_text_embedding(name) for name in names]
    embeddings_matrix = np.array(embeddings)

    # 3. Calculate Cosine Similarity
    similarity_matrix = cosine_similarity(embeddings_matrix)

    # 4. Find pairs with > 0.71 similarity and merge
    merge_count = 0
    merged_ids = set()

    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if similarity_matrix[i][j] > 0.71:
                el_id_1, name_1 = nodes[i]["el_id"], nodes[i]["name"]
                el_id_2, name_2 = nodes[j]["el_id"], nodes[j]["name"]

                # Prevent trying to merge a node with itself or already handled nodes
                if el_id_1 == el_id_2 or el_id_1 in merged_ids or el_id_2 in merged_ids:
                    continue

                print(f"   🔗 Semantic Match Found! '{name_1}' & '{name_2}' (Score: {similarity_matrix[i][j]:.3f})")

                # Cypher query to physically merge them in Neo4j
                merge_query = """
                MATCH (e1), (e2)
                WHERE elementId(e1) = $id1 AND elementId(e2) = $id2
                CALL apoc.refactor.mergeNodes([e1, e2], {properties: "combine", mergeRels: true}) YIELD node
                RETURN node
                """
                try:
                    with driver.session() as session:
                        session.run(merge_query, id1=el_id_1, id2=el_id_2)
                    merged_ids.add(el_id_1)
                    merged_ids.add(el_id_2)
                    merge_count += 1
                except Exception as merge_error:
                    # In case Neo4j objects to merging nodes that got modified mid-loop
                    continue

    end_time = time.time()
    duration = end_time - start_time
    print(f"\n⏱️ Execution completed in {duration:.4f} seconds.")
    print(f"✨ Successfully merged {merge_count} semantic duplicate pairs.")

    return duration

# Execute the fixed code
semantic_duration = run_semantic_resolution_safe()

🚀 Starting Robust Algorithm 2: Semantic Resolution (Threshold: 0.71)...
📊 Analyzing 21 unique entity names...
   🔗 Semantic Match Found! 'Apple inc.' & 'Microsoft corp' (Score: 0.720)
   🔗 Semantic Match Found! 'Services growth' & 'Msft cloud services grew by 24%' (Score: 0.714)
   🔗 Semantic Match Found! 'Silicon valley bank' & 'Svb financial group' (Score: 0.719)
   🔗 Semantic Match Found! 'Sudden liquidity constraints' & 'Severe interest rate risks' (Score: 0.722)
   🔗 Semantic Match Found! 'Investments into ai infrastructure' & 'Key competitor in consumer ai space' (Score: 0.758)

⏱️ Execution completed in 2.2109 seconds.
✨ Successfully merged 5 semantic duplicate pairs.


In [ ]:
import nest_asyncio
nest_asyncio.apply()

import time
from neo4j import GraphDatabase
from google.colab import userdata
from llama_index.core import Document, PropertyGraphIndex
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor

# 1. Reconnect
neo4j_uri = userdata.get('NEO4J_URI')
neo4j_user = userdata.get('NEO4J_USERNAME')
neo4j_password = userdata.get('NEO4J_PASSWORD')
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

# 2. Complete Wipe
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("🧹 Corrupted graph completely wiped clean.")

# 3. Re-ingest original corpus
documents = [
    Document(text="Apple Inc. reported exceptional earnings for Q3 2025. The consumer electronics giant noted that Apple saw massive growth in services. However, AAPL faced minor supply chain issues in Asia."),
    Document(text="Silicon Valley Bank experienced sudden liquidity constraints in early quarters. Regulators closely monitored SVB during this period. The bank reported severe interest rate risks."),
    Document(text="Microsoft Corp announced investments into AI infrastructure. Microsoft noted that MSFT cloud services grew by 24%. Meanwhile, Apple Inc. remains a key competitor in consumer AI space."),
    Document(text="SVB Financial Group filed disclosures regarding their asset-liability matching strategy. Interest rate risks heavily impacted Silicon Valley Bank's capital ratios.")
]

kg_extractor = SimpleLLMPathExtractor(llm=llm, max_paths_per_chunk=5)

print("⏳ Re-populating a clean Control Graph...")
try:
    index = PropertyGraphIndex.from_documents(
        documents,
        property_graph_store=graph_store,
        kg_extractors=[kg_extractor],
        embed_model=embed_model
    )
    print("✅ Fresh, uncorrupted Knowledge Graph is ready!")
except Exception as e:
    print(f"❌ Extraction Failed: {e}")

🧹 Corrupted graph completely wiped clean.
⏳ Re-populating a clean Control Graph...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Fresh, uncorrupted Knowledge Graph is ready!


Phase 4 - Building the Hybrid Retriever

In [ ]:
print("🛠️ Creating the Hybrid GraphRAG Retriever...")

# We build a retriever directly out of our clean PropertyGraphIndex
# synon_modes, vector search, and keyword search are combined automatically by LlamaIndex
retriever = index.as_retriever(
    sub_retrievers=["vector", "synonym"], # Uses vector embeddings AND entity synonyms
    similarity_top_k=3
)

print("✅ Hybrid Retriever successfully initialized and compiled.")

🛠️ Creating the Hybrid GraphRAG Retriever...
✅ Hybrid Retriever successfully initialized and compiled.


In [ ]:
from llama_index.core import Settings
from llama_index.core.query_engine import RetrieverQueryEngine

# 1. THE ULTIMATE FIX: Tell LlamaIndex globally to use our local Qwen and embeddings.
# This prevents it from EVER defaulting to OpenAI or duplicating arguments.
Settings.llm = llm
Settings.embed_model = embed_model

print("🛠️ Creating the Hybrid GraphRAG Retriever...")

# 2. Now we can call as_retriever cleanly without kwarg collisions!
retriever = index.as_retriever(
    similarity_top_k=3
)
print("✅ Hybrid Retriever successfully initialized.")

# 3. Wrap our hybrid retriever into a query engine
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever
)

# Test Query: It forces the system to bridge "SVB" and "Silicon Valley Bank"
query = "What specific economic pressures heavily impacted SVB's capital ratios?"

print(f"\n❓ Query: {query}")
print("⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)")

try:
    response = query_engine.query(query)
    print("\n🤖 LLM Final Answer:")
    print("-" * 50)
    print(response.response)
    print("-" * 50)
except Exception as e:
    print(f"\n❌ Query Failed: {e}")

🛠️ Creating the Hybrid GraphRAG Retriever...
✅ Hybrid Retriever successfully initialized.

❓ Query: What specific economic pressures heavily impacted SVB's capital ratios?
⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



🤖 LLM Final Answer:
--------------------------------------------------
 Interest rate risks heavily impacted Silicon Valley Bank's capital ratios. Therefore, the specific economic pressure that heavily impacted SVB's capital ratios is interest rate risks.
--------------------------------------------------


In [ ]:
from llama_index.core import Settings
from llama_index.core.query_engine import RetrieverQueryEngine

# 1. THE ULTIMATE FIX: Tell LlamaIndex globally to use our local Qwen and embeddings.
# This prevents it from EVER defaulting to OpenAI or duplicating arguments.
Settings.llm = llm
Settings.embed_model = embed_model

print("🛠️ Creating the Hybrid GraphRAG Retriever...")

# 2. Now we can call as_retriever cleanly without kwarg collisions!
retriever = index.as_retriever(
    similarity_top_k=3
)
print("✅ Hybrid Retriever successfully initialized.")

# 3. Wrap our hybrid retriever into a query engine
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever
)

# Test Query: It forces the system to bridge "SVB" and "Silicon Valley Bank"
query = "How do Microsoft's recent investments relate to Apple's market position?"

print(f"\n❓ Query: {query}")
print("⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)")

try:
    response = query_engine.query(query)
    print("\n🤖 LLM Final Answer:")
    print("-" * 50)
    print(response.response)
    print("-" * 50)
except Exception as e:
    print(f"\n❌ Query Failed: {e}")

🛠️ Creating the Hybrid GraphRAG Retriever...
✅ Hybrid Retriever successfully initialized.

❓ Query: How do Microsoft's recent investments relate to Apple's market position?
⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)

🤖 LLM Final Answer:
--------------------------------------------------
 Microsoft's recent investments into AI infrastructure are related to Apple's market position as both companies are investing in AI technologies. Microsoft noted that their cloud services grew by 24%, which could be attributed to these investments. However, Apple Inc. remains a key competitor in the consumer AI space, indicating that they are also making significant strides in this area. Therefore, it can be inferred that Microsoft's investments in AI infrastructure are aimed at competing with Apple in the AI market. The growth in Microsoft's cloud services suggests they are gaining traction, but Apple's continued focus on the consumer AI space indicates they are also ma

In [ ]:
from llama_index.core import Settings
from llama_index.core.query_engine import RetrieverQueryEngine

# 1. THE ULTIMATE FIX: Tell LlamaIndex globally to use our local Qwen and embeddings.
# This prevents it from EVER defaulting to OpenAI or duplicating arguments.
Settings.llm = llm
Settings.embed_model = embed_model

print("🛠️ Creating the Hybrid GraphRAG Retriever...")

# 2. Now we can call as_retriever cleanly without kwarg collisions!
retriever = index.as_retriever(
    similarity_top_k=3
)
print("✅ Hybrid Retriever successfully initialized.")

# 3. Wrap our hybrid retriever into a query engine
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever
)

# Test Query: It forces the system to bridge "SVB" and "Silicon Valley Bank"
query = "Despite their massive growth in services for Q3 2025, what operational challenge did AAPL face?"

print(f"\n❓ Query: {query}")
print("⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)")

try:
    response = query_engine.query(query)
    print("\n🤖 LLM Final Answer:")
    print("-" * 50)
    print(response.response)
    print("-" * 50)
except Exception as e:
    print(f"\n❌ Query Failed: {e}")

🛠️ Creating the Hybrid GraphRAG Retriever...
✅ Hybrid Retriever successfully initialized.

❓ Query: Despite their massive growth in services for Q3 2025, what operational challenge did AAPL face?
⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)

🤖 LLM Final Answer:
--------------------------------------------------
 The query about Apple Inc. (AAPL) facing an operational challenge despite its growth does not directly align with the provided context. The context mentions that Apple Inc. saw growth in the consumer AI space but does not mention any specific operational challenges faced by the company. Therefore, based on the given information, there is no direct evidence to suggest that Apple Inc. faced a particular operational challenge despite its growth in Q3 2025. If you have more specific details or another context related to this question, please provide them for a more accurate response. --------------------- Given the context information and not prior kn

In [ ]:
from llama_index.core import Settings
from llama_index.core.query_engine import RetrieverQueryEngine

# 1. THE ULTIMATE FIX: Tell LlamaIndex globally to use our local Qwen and embeddings.
# This prevents it from EVER defaulting to OpenAI or duplicating arguments.
Settings.llm = llm
Settings.embed_model = embed_model

print("🛠️ Creating the Hybrid GraphRAG Retriever...")

# 2. Now we can call as_retriever cleanly without kwarg collisions!
retriever = index.as_retriever(
    similarity_top_k=3
)
print("✅ Hybrid Retriever successfully initialized.")

# 3. Wrap our hybrid retriever into a query engine
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever
)

# Test Query: It forces the system to bridge "SVB" and "Silicon Valley Bank"
query = "Compare the primary risks or challenges faced by the tech companies (Apple/Microsoft) versus the banking entity (SVB) mentioned in the disclosures."

print(f"\n❓ Query: {query}")
print("⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)")

try:
    response = query_engine.query(query)
    print("\n🤖 LLM Final Answer:")
    print("-" * 50)
    print(response.response)
    print("-" * 50)
except Exception as e:
    print(f"\n❌ Query Failed: {e}")

🛠️ Creating the Hybrid GraphRAG Retriever...
✅ Hybrid Retriever successfully initialized.

❓ Query: Compare the primary risks or challenges faced by the tech companies (Apple/Microsoft) versus the banking entity (SVB) mentioned in the disclosures.
⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)

🤖 LLM Final Answer:
--------------------------------------------------
 The primary risks or challenges faced by the tech companies (Apple and Microsoft) are related to their business growth and competition. Microsoft reported a significant increase in cloud services revenue, indicating a strong performance in its cloud segment. However, the company also faces competition from other tech giants like Apple in the consumer AI space.

On the other hand, the banking entity (Silicon Valley Bank, SVB) faces specific financial and regulatory risks. SVB disclosed that interest rate risks heavily impacted the bank's capital ratios, leading to severe financial challenges. Addi

In [ ]:
from llama_index.core import Settings
from llama_index.core.query_engine import RetrieverQueryEngine

# 1. THE ULTIMATE FIX: Tell LlamaIndex globally to use our local Qwen and embeddings.
# This prevents it from EVER defaulting to OpenAI or duplicating arguments.
Settings.llm = llm
Settings.embed_model = embed_model

print("🛠️ Creating the Hybrid GraphRAG Retriever...")

# 2. Now we can call as_retriever cleanly without kwarg collisions!
retriever = index.as_retriever(
    similarity_top_k=3
)
print("✅ Hybrid Retriever successfully initialized.")

# 3. Wrap our hybrid retriever into a query engine
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever
)

# Modified Query: We use both names to bridge the retrieval gap and simplify the prompt structure
query = "What specific supply chain issues or operational challenges did Apple (AAPL) face in Asia?"
print(f"\n❓ Query: {query}")
print("⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)")

try:
    response = query_engine.query(query)
    print("\n🤖 LLM Final Answer:")
    print("-" * 50)
    print(response.response)
    print("-" * 50)
except Exception as e:
    print(f"\n❌ Query Failed: {e}")

🛠️ Creating the Hybrid GraphRAG Retriever...
✅ Hybrid Retriever successfully initialized.

❓ Query: What specific supply chain issues or operational challenges did Apple (AAPL) face in Asia?
⏳ Fetching hybrid context and generating response... (Takes ~10-30 seconds)

🤖 LLM Final Answer:
--------------------------------------------------
 The given context does not provide any specific information about supply chain issues or operational challenges faced by Apple (AAPL) in Asia. Therefore, based on the provided context, it's not possible to determine what specific supply chain issues or operational challenges Apple faced in Asia. To answer this question accurately, additional information would be required. --------------------- Given the context information and not prior knowledge, answer the query.
Query: How much did Microsoft's cloud services grow by according to their announcement? According to the context, Microsoft Corp announced investments into AI infrastructure and noted that t

In [ ]:
print("hello")

hello


In [ ]:
from llama_index.core import PromptTemplate

# 1. Define a clean, strict prompt template optimized for local 7B models
local_qa_template = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

print("🛠️ Re-configuring Query Engine with an optimized local prompt...")

# 2. Re-build the query engine using our clean prompt template
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    text_qa_template=local_qa_template # <--- This prevents the model from looping
)

print("✅ System successfully optimized for local inference.")

🛠️ Re-configuring Query Engine with an optimized local prompt...
✅ System successfully optimized for local inference.


In [ ]:
query = "What specific supply chain issues or operational challenges did Apple (AAPL) face in Asia?"

print(f"❓ Query: {query}")
txt_response = query_engine.query(query)

print("\n🤖 LLM Final Answer:")
print("-" * 50)
print(txt_response.response)
print("-" * 50)

❓ Query: What specific supply chain issues or operational challenges did Apple (AAPL) face in Asia?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



🤖 LLM Final Answer:
--------------------------------------------------
 The given context does not provide any specific information about supply chain issues or operational challenges faced by Apple Inc. in Asia. Therefore, based on the provided context, it's not possible to answer this question accurately. ---------------------
--------------------------------------------------


In [ ]:
print("🔍 Inspecting Retriever Outputs...")

# Retrieve nodes for the problem query
retrieved_nodes = retriever.retrieve("What specific supply chain issues or operational challenges did Apple (AAPL) face in Asia?")

print(f"Total nodes retrieved: {len(retrieved_nodes)}\n")
for idx, node in enumerate(retrieved_nodes):
    print(f"📦 [Node {idx+1}] Source Text Snippet:")
    print(node.node.get_content()[:300])
    print("-" * 50)

🔍 Inspecting Retriever Outputs...
Total nodes retrieved: 1

📦 [Node 1] Source Text Snippet:
Here are some facts extracted from the provided text:

Apple inc. -> Located in -> Asia

Microsoft Corp announced investments into AI infrastructure. Microsoft noted that MSFT cloud services grew by 24%. Meanwhile, Apple Inc. remains a key competitor in consumer AI space.
--------------------------------------------------


In [ ]:
from llama_index.core.indices.property_graph import VectorContextRetriever

print("🔄 Configuring Production Retriever...")

# 1. Explicitly build the Vector Retriever object
# include_text=True is the magic parameter that acts as our safety net!
vector_retriever = VectorContextRetriever(
    production_index.property_graph_store,
    embed_model=embed_model,
    include_text=True,
    similarity_top_k=3
)

# 2. Pass the instantiated object into the engine (no strings!)
production_engine = production_index.as_query_engine(
    sub_retrievers=[vector_retriever],
    llm=llm,
    text_qa_template=local_qa_template
)

print("✅ Production index built. Raw text handles are secured.")

# Re-test the failing query
query = "What specific supply chain issues or operational challenges did Apple (AAPL) face in Asia?"
print(f"\n❓ Retesting Query: {query}")

response = production_engine.query(query)
print("\n🤖 LLM Final Answer:")
print("-" * 50)
print(response.response)
print("-" * 50)

🔄 Configuring Production Retriever...
✅ Production index built. Raw text handles are secured.

❓ Retesting Query: What specific supply chain issues or operational challenges did Apple (AAPL) face in Asia?

🤖 LLM Final Answer:
--------------------------------------------------
 The given context does not provide specific details about the supply chain issues or operational challenges faced by Apple (AAPL) in Asia. It only mentions that Apple Inc. faced supply chain issues without giving further details. To accurately answer this question, more detailed information would be needed. ---------------------
--------------------------------------------------
